In [ ]:
# -----------------------------------------------------------------------------
# Install Dependencies
# -----------------------------------------------------------------------------
!pip install -q \
    langchain-community==0.3.27 \
    chromadb==1.0.15 \
    pymupdf==1.26.3 \
    pypdf \
    gdown \
    langchain-groq \
    sentence_transformers \
    google.colab


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.

In [ ]:
"""
================================================================================
Script Name    : AI_Project_1_JasonWayneMcGinty.py
University     : University of Texas at Austin
Program        : AI Agents in Business Applications
Author         : Jason Wayne McGinty
Date Created   : 2026-04-02
Last Modified  : 2026-04-05
Version        : Google Colab 3.12.13
Description    : This brief program demonstrates how to load data into a vector
                 database using LangChain and ChromaDB. The program then uses
                 Retrieval-Augmented Generation (RAG) to answer questions about
                 the data
Copyright      : © 2026 Jason Wayne McGinty
License        : MIT License
================================================================================

VERSION HISTORY
--------------------------------------------------------------------------------
| Version | Date       | Author         | Changes                              |
|---------|------------|----------------|--------------------------------------|
| 1.0.0   | 2026-04-02 | Jason McGinty  | Initial release                      |
| 1.0.1   | 2026-04-03 | Jason McGinty  | Classes                              |
| 1.0.2   | 2026-04-04 | Jason McGinty  | Comments                             |
| 1.0.3   | 2026-04-05 | Jason McGinty  | PEP 8 naming conventions             |
--------------------------------------------------------------------------------

================================================================================
AI ENABLED
================================================================================
This code was developed with assistance from Lumo AI (Proton AG).
AI assisted with:
- Implementing best practices (PEP 8)
- Error handling patterns

All final implementation decisions were made by the author.
================================================================================

WORKS CITED (MLA FORMAT)
--------------------------------------------------------------------------------
Groq. "GroqCloud API Documentation." Groq, console.groq.com/docs, 2024.
Hugging Face. "Sentence Transformers." Hugging Face,
    huggingface.co/sentence-transformers/all-MiniLM-L6-v2, 2023.
LangChain. "RetrievalQA Chain." LangChain Documentation,
    python.langchain.com/docs/use_cases/question_answering/quickstart, 2024.
Meta. "Llama 3." Meta AI, ai.meta.com/llama/, 2024.
Podolny, Joel M., and Morten T. Hansen. "How Apple Is Organized for Innovation."
    Harvard Business Review, vol. 99, no. 3, May-June 2021, pp. 72-81.
Google. "gdown: Download Files from Google Drive." GitHub,
    github.com/wkentaro/gdown, 2023.
Proton. "Lumo AI Assistant." Proton AG, lumo.proton.me, 2026.
"""

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
import sys
import gdown
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb
from langchain_community.vectorstores import Chroma
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from google.colab import userdata
from dataclasses import dataclass
from datetime import datetime

# -----------------------------------------------------------------------------
# Logging
# -----------------------------------------------------------------------------
def print_log(message, level="INFO"):
    """Prints a message with a timestamp."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] [{level}] {message}")

# -----------------------------------------------------------------------------
# Configuration Constants
# -----------------------------------------------------------------------------
FILE_ID = '1t-s1nXALzl_fMuvNm6yKzEUCmv_0croB'
OUTPUT_PATH = 'How_Apple_is_Organized_for_Innovation.pdf'
DEFAULT_CHUNK_SIZE = 1000
DEFAULT_CHUNK_OVERLAP = 200
DEFAULT_TEMPERATURE = 0.0
DEFAULT_SEARCH_ARGS = 3

# -----------------------------------------------------------------------------
# Class RAGConfig
# -----------------------------------------------------------------------------
@dataclass
class RAGConfig:
    """Configuration settings for the RAG pipeline."""
    file_id: str
    output_path: str
    chunk_size: int
    chunk_overlap: int
    embedding_model_name: str      # e.g., "all-MiniLM-L6-v2"
    llm_model_name: str            # e.g., "llama-3.3-70b-versatile"
    persist_directory: str
    collection_name: str
    temperature: float
    search_args: int
    groq_api_key_secret: str       # Name of the secret in Colab

# -----------------------------------------------------------------------------
# Class MediaDownload
# -----------------------------------------------------------------------------
class MediaDownload:
    """Handles downloading files from Google Drive."""

    def __init__(self, file_id, output_path):
        self.file_id = file_id
        self.output_path = output_path

    def download_pdf(self):
        """Downloads a PDF file from Google Drive."""
        try:
            gdown.download(id=self.file_id, output=self.output_path, quiet=True)
            print_log("Download successful.", "SUCCESS")
        except Exception as e:
            print_log(f"Error downloading file: {e}", "ERROR")
        finally:
            print_log("Media download process finished.", "INFO")

# -----------------------------------------------------------------------------
# Class MediaLoader
# -----------------------------------------------------------------------------
class MediaLoader:
    """Handles loading PDF documents."""

    def __init__(self, output_path):
        self.output_path = output_path

    def load_pdf(self):
        """Loads a PDF file and returns document objects."""
        try:
            loader = PyPDFLoader(self.output_path)
            document = loader.load()
            print_log(f"Successfully loaded {len(document)} pages from the PDF.", "SUCCESS")
        except Exception as e:
            print_log(f"Error loading media file: {e}", "ERROR")
            document = None
        finally:
            print_log("Media loading process finished.", "INFO")
        return document

# -----------------------------------------------------------------------------
# Class DataChunking
# -----------------------------------------------------------------------------
class DataChunking:
    """Handles splitting documents into chunks."""

    def __init__(self, chunk_size, chunk_overlap):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def split_documents(self, document):
        """Splits documents into smaller chunks for embedding."""
        try:
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size,
                chunk_overlap=self.chunk_overlap
            )
            chunks = text_splitter.split_documents(document)
            print_log(f"Successfully created {len(chunks)} chunks.", "SUCCESS")
        except Exception as e:
            print_log(f"Error chunking media file: {e}", "ERROR")
            chunks = None
        finally:
            print_log("Media chunking process finished.", "INFO")
        return chunks

# -----------------------------------------------------------------------------
# Class ChromaDB
# -----------------------------------------------------------------------------
class ChromaDB:
    """Handles ChromaDB vector database operations."""

    def __init__(self, persist_directory):
        self.persist_directory = persist_directory
        self.client = None

    def initialize(self):
        """Initializes the ChromaDB client."""
        try:
            self.client = chromadb.Client()
            print_log("Chroma vector database initialized.", "SUCCESS")
        except Exception as e:
            print_log(f"Error initializing Chroma vector database: {e}", "ERROR")
        finally:
            print_log("Chroma vector database initialization process finished.", "INFO")
        return self.client

# -----------------------------------------------------------------------------
# Class EmbeddingModel
# -----------------------------------------------------------------------------
class EmbeddingModel:
    """Handles embedding documents into a vector store."""

    def __init__(self, chunks, model_name, client, collection_name):
        self.chunks = chunks
        self.model_name = model_name
        self.client = client
        self.collection_name = collection_name
        self.vectorstore = None

    def create_vectorstore(self):
        """Creates a vector store from document chunks."""
        try:
            # embedding_instance = HuggingFaceEmbeddings(model_name=self.model_name)
            embedding_instance = SentenceTransformerEmbeddings(model_name=self.model_name)
            print_log("Embedding model initialized.", "SUCCESS")

            self.vectorstore = Chroma.from_documents(
                documents=self.chunks,
                embedding=embedding_instance,
                client=self.client,
                collection_name=self.collection_name
            )
            print_log("Chunks successfully embedded and stored in Chroma vector database.", "SUCCESS")
        except Exception as e:
            print_log(f"Error initializing embedding model: {e}", "ERROR")
        finally:
            print_log("Embedding model initialization process finished.", "INFO")
        return self.vectorstore

# -----------------------------------------------------------------------------
# Class GroqLLM
# -----------------------------------------------------------------------------
class GroqLLM:
    """Handles Groq LLM initialization and interaction."""

    def __init__(self, model, temperature, api_key):
        self.model = model
        self.temperature = temperature
        self.api_key = api_key
        self.llm = None

    def initialize(self):
        """Initializes the Groq LLM client."""
        try:
            self.llm = ChatGroq(
                model=self.model,
                temperature=self.temperature,
                api_key=self.api_key
            )
            print_log(f"Groq LLM initialized ({self.model}).", "SUCCESS")
        except Exception as e:
            print_log(f"Error initializing Groq LLM: {e}", "ERROR")
        finally:
            print_log("Groq LLM initialization process finished.", "INFO")
        return self.llm

    def execute(self):
        """Executes the query directly against the LLM."""
        try:
            response = self.llm.invoke(self.query)
            self.answer = response.content

            print_log(f"❓ Q (LLM Only): {self.query}", "QUERY")
            print_log(f"✅ A (LLM Only): {self.answer}", "SUCCESS")
            print()

        except Exception as e:
            print_log(f"Error querying LLM directly: {e}", "ERROR")
        finally:
            print_log("Direct LLM query process finished.", "INFO")
        return self.answer

# -----------------------------------------------------------------------------
# Class DirectLLMQuery
# -----------------------------------------------------------------------------
class DirectLLMQuery:
    """Handles querying the LLM directly without RAG."""

    def __init__(self, llm, query):
        self.llm = llm
        self.query = query
        self.answer = None

    def execute(self):
        """Executes the query directly against the LLM."""
        try:
            response = self.llm.invoke(self.query)
            self.answer = response.content

            print_log(f"❓ Q (LLM Only): {self.query}", "QUERY")
            print_log(f"✅ A (LLM Only): {self.answer}", "SUCCESS")
            print()

        except Exception as e:
            print_log(f"Error querying LLM directly: {e}", "ERROR")
        finally:
            print_log("Direct LLM query process finished.", "INFO")
        return self.answer

# -----------------------------------------------------------------------------
# Class Retriever
# -----------------------------------------------------------------------------
class Retriever:
    """Handles document retrieval from the vector store."""

    def __init__(self, vectorstore, search_args):
        self.vectorstore = vectorstore
        self.search_args = search_args
        self.retriever = None

    def initialize(self):
        """Creates a retriever from the vector store."""
        try:
            self.retriever = self.vectorstore.as_retriever(
                search_kwargs={"k": self.search_args}
            )
            print_log(f"Retriever initialized (k={self.search_args}).", "SUCCESS")
        except Exception as e:
            print_log(f"Error initializing Retriever: {e}", "ERROR")
        finally:
            print_log("Retriever initialization process finished.", "INFO")
        return self.retriever

# -----------------------------------------------------------------------------
# Class RAGChain
# -----------------------------------------------------------------------------
class RAGChain:
    """Handles the RAG question-answering chain."""

    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever
        self.qa_chain = None

    def initialize(self):
        """Builds the RetrievalQA chain."""
        try:
            self.qa_chain = RetrievalQA.from_chain_type(
                llm=self.llm,
                chain_type="stuff",
                retriever=self.retriever,
                return_source_documents=True
            )
            print_log("RAG chain initialized.", "SUCCESS")
        except Exception as e:
            print_log(f"Error initializing RAG chain: {e}", "ERROR")
        finally:
            print_log("RAG chain initialization process finished.", "INFO")
        return self.qa_chain

# -----------------------------------------------------------------------------
# Class RAGQueryAnswer
# -----------------------------------------------------------------------------
class RAGQueryAnswer:
    """Handles querying the RAG chain and displaying results."""

    def __init__(self, qa_chain, query):
        self.qa_chain = qa_chain
        self.query = query
        self.answer = None
        self.raw_result = None

    def execute(self):
        """Executes the query and prints the answer."""
        try:
            self.raw_result = self.qa_chain.invoke({"query": self.query})

            if isinstance(self.raw_result, dict) and "result" in self.raw_result:
                self.answer = self.raw_result["result"]
            else:
                self.answer = str(self.raw_result)

            print_log(f"❓ Q (RAG): {self.query}", "QUERY")
            print_log(f"🤖 A (RAG): {self.answer}", "SUCCESS")
            print()

        except Exception as e:
            print_log(f"Error answering query: {e}", "ERROR")
        finally:
            print_log("Query and answering process finished.", "INFO")
        return self.answer

# -----------------------------------------------------------------------------
# Main Program
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    try:
        print_log("Starting program...", "INFO")

        # -------------------------------------------------------------------------
        # Configuration
        # -------------------------------------------------------------------------
        config = RAGConfig(
            file_id=FILE_ID,
            output_path=OUTPUT_PATH,
            chunk_size=DEFAULT_CHUNK_SIZE,
            chunk_overlap=DEFAULT_CHUNK_OVERLAP,
            embedding_model_name="all-MiniLM-L6-v2",
            llm_model_name="llama-3.3-70b-versatile",
            persist_directory='./chroma_db',
            collection_name="apple_rag_collection",
            temperature=DEFAULT_TEMPERATURE,
            search_args=DEFAULT_SEARCH_ARGS,
            groq_api_key_secret="GROQ_API_KEY"
        )

        # -------------------------------------------------------------------------
        # Step 00: Download Media
        # -------------------------------------------------------------------------
        media = MediaDownload(config.file_id, config.output_path)
        media.download_pdf()

        # -------------------------------------------------------------------------
        # Step 01: Load Media
        # -------------------------------------------------------------------------
        loader = MediaLoader(config.output_path)
        document = loader.load_pdf()
        if not document:
            raise Exception("Document loading failed. Stopping program.")

        # -------------------------------------------------------------------------
        # Step 02: Chunk Data
        # -------------------------------------------------------------------------
        chunker = DataChunking(config.chunk_size, config.chunk_overlap)
        chunks = chunker.split_documents(document)
        if not chunks:
            raise Exception("Chunking failed. Stopping program.")

        # -------------------------------------------------------------------------
        # Step 03: Initialize ChromaDB
        # -------------------------------------------------------------------------
        chroma_db = ChromaDB(config.persist_directory)
        chroma_client = chroma_db.initialize()
        if not chroma_client:
            raise Exception("ChromaDB initialization failed. Stopping program.")

        # -------------------------------------------------------------------------
        # Step 04: Create Embeddings
        # -------------------------------------------------------------------------
        embedding_model = EmbeddingModel(
            chunks=chunks,
            model_name=config.embedding_model_name,
            client=chroma_client,
            collection_name=config.collection_name
        )
        vectorstore = embedding_model.create_vectorstore()
        if not vectorstore:
            raise Exception("Embedding failed. Stopping program.")

        # -------------------------------------------------------------------------
        # Step 05: Initialize LLM
        # -------------------------------------------------------------------------
        api_key = userdata.get(config.groq_api_key_secret)
        if not api_key:
            raise ValueError(f"API Key '{config.groq_api_key_secret}' not found in Colab Secrets.")

        groq_llm = GroqLLM(
            model=config.llm_model_name,
            temperature=config.temperature,
            api_key=api_key
        )
        llm = groq_llm.initialize()
        if not llm:
            raise Exception("LLM initialization failed. Stopping program.")

        # -------------------------------------------------------------------------
        # Step 07: Initialize Retriever
        # -------------------------------------------------------------------------
        retriever_instance = Retriever(vectorstore, config.search_args)
        retriever = retriever_instance.initialize()
        if not retriever:
            raise Exception("Retriever initialization failed. Stopping program.")

        # -------------------------------------------------------------------------
        # Step 08: Initialize RAG Chain
        # -------------------------------------------------------------------------
        rag_chain_instance = RAGChain(llm, retriever)
        qa_chain = rag_chain_instance.initialize()
        if not qa_chain:
            raise Exception("RAG chain initialization failed. Stopping program.")

        # -------------------------------------------------------------------------
        # Step 09: Execute Queries
        # -------------------------------------------------------------------------
        query_list = [
            {
                "id": 1,
                "query": "Who are the authors of this article and who published this article?"
            },
            {
                "id": 2,
                "query": "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
            },
            {
                "id": 3,
                "query": "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"
            }
        ]

        # Run against LLM only
        for query_item in query_list:
          print_log("Starting batch queries with Groq LLM...", "INFO")
          direct_query = DirectLLMQuery(llm, query_item["query"])
          direct_query.execute()

        # Answer the questions using the RAG Chain
        print_log("Starting batch queries with RAG...", "INFO")
        for query_item in query_list:
            query_answer = RAGQueryAnswer(qa_chain, query_item["query"])
            query_answer.execute()

        print_log("Program completed successfully.", "SUCCESS")

    except Exception as e:
        print_log(f"❌ FATAL ERROR: {e}", "ERROR")
        print_log("Program terminated due to errors.", "ERROR")
        sys.exit(1)
    finally:
        print_log("Program finished.", "INFO")

[2026-04-05 16:14:56] [INFO] Starting program...
[2026-04-05 16:14:58] [SUCCESS] Download successful.
[2026-04-05 16:14:58] [INFO] Media download process finished.
[2026-04-05 16:14:58] [SUCCESS] Successfully loaded 11 pages from the PDF.
[2026-04-05 16:14:58] [INFO] Media loading process finished.
[2026-04-05 16:14:58] [SUCCESS] Successfully created 49 chunks.
[2026-04-05 16:14:58] [INFO] Media chunking process finished.
[2026-04-05 16:14:58] [SUCCESS] Chroma vector database initialized.
[2026-04-05 16:14:58] [INFO] Chroma vector database initialization process finished.


/tmp/ipykernel_13547/3688768750.py:203: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_instance = SentenceTransformerEmbeddings(model_name=self.model_name)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datas

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[2026-04-05 16:15:13] [SUCCESS] Embedding model initialized.
[2026-04-05 16:15:14] [SUCCESS] Chunks successfully embedded and stored in Chroma vector database.
[2026-04-05 16:15:14] [INFO] Embedding model initialization process finished.
[2026-04-05 16:15:15] [SUCCESS] Groq LLM initialized (llama-3.3-70b-versatile).
[2026-04-05 16:15:15] [INFO] Groq LLM initialization process finished.
[2026-04-05 16:15:15] [SUCCESS] Retriever initialized (k=3).
[2026-04-05 16:15:15] [INFO] Retriever initialization process finished.
[2026-04-05 16:15:15] [SUCCESS] RAG chain initialized.
[2026-04-05 16:15:15] [INFO] RAG chain initialization process finished.
[2026-04-05 16:15:15] [INFO] Starting batch queries with Groq LLM...
[2026-04-05 16:15:15] [QUERY] ❓ Q (LLM Only): Who are the authors of this article and who published this article?
[2026-04-05 16:15:15] [SUCCESS] ✅ A (LLM Only): This conversation has just begun. You haven't provided an article, so I'm not aware of any authors or publishers. If you

WORKS CITED (MLA FORMAT)
--------------------------------------------------------------------------------
LangChain. "RetrievalQA Chain." LangChain Documentation,
    python.langchain.com/docs/use_cases/question_answering/quickstart, 2024.

LlamaIndex. "ChromaDB Vector Store." LlamaIndex Documentation,
    docs.llamaindex.ai/en/stable/module_guides/deploying/chroma.html, 2024.

Meta. "Llama 3." Meta AI, ai.meta.com/llama/, 2024.

Podolny, Joel M., and Morten T. Hansen. "How Apple Is Organized for Innovation."
    Harvard Business Review, vol. 99, no. 3, May-June 2021, pp. 72-81.

Groq. "GroqCloud API Documentation." Groq, console.groq.com/docs, 2024.

Hugging Face. "Sentence Transformers." Hugging Face,
    huggingface.co/sentence-transformers/all-MiniLM-L6-v2, 2023.

Google. "gdown: Download Files from Google Drive." GitHub,
    github.com/wkentaro/gdown, 2023.

Proton. "Lumo AI Assistant." Proton AG, lumo.proton.me, 2026.